In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType)

spark = (
    SparkSession.builder
    .appName("bdpa-clickflow-eclothing-recommender")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.memory.fraction", "0.6")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

raw = (
    spark.read
        .option("sep", ";")
        .option("header", "true")
        .csv("../dataset/e_shop_clothing_2008.csv")
        .toDF(
              "year", "month", "day", "order", "country",
        "session_id", "category", "item_id", "colour",
        "location", "photo_type", "price", "price_above_avg", "page"
        )
        .withColumn("year",            F.col("year").cast(IntegerType()))
        .withColumn("month",           F.col("month").cast(IntegerType()))
        .withColumn("day",             F.col("day").cast(IntegerType()))
        .withColumn("order",           F.col("order").cast(IntegerType()))
        .withColumn("country",         F.col("country").cast(IntegerType()))
        .withColumn("session_id",      F.col("session_id").cast(IntegerType()))
        .withColumn("category",        F.col("category").cast(IntegerType()))
        .withColumn("colour",          F.col("colour").cast(IntegerType()))
        .withColumn("location",        F.col("location").cast(IntegerType()))
        .withColumn("photo_type",      F.col("photo_type").cast(IntegerType()))
        .withColumn("price",           F.col("price").cast(IntegerType()))
        .withColumn("price_above_avg", F.col("price_above_avg").cast(IntegerType()))
        .withColumn("page",            F.col("page").cast(IntegerType()))
) 
    

assert raw.count() == 165474
assert raw.filter(F.col("session_id").isNull()).count() == 0
assert raw.filter(F.col("item_id").isNull()).count() == 0
assert raw.select("month").distinct().count() == 5 # april to august
assert raw.select("category").distinct().count() == 4 # trousers, skirts, blouses, sale
assert raw.select("item_id").distinct().count() == 217


print("All validation checks passed.")
raw.printSchema()
raw.show(5)



All validation checks passed.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- item_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photo_type: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_avg: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|year|month|day|order|country|session_id|category|item_id|colour|location|photo_type|price|price_above_avg|page|
+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|2008|    4|  1|    1|     29|         1|       1|    A13|     1|  

In [13]:
session_lengths = (
    raw.groupBy("session_id")
        .agg(F.count("*").alias("n_clicks"))
)

session_lengths.describe("n_clicks").show()

n_single = session_lengths.filter(F.col("n_clicks") == 1).count()

n_total = session_lengths.count() 
print(f"Single click session: {n_single}/{n_total} ({(n_single/n_total * 100)} %)")


# item popularity

item_freq = (
    raw.groupBy('item_id') 
        .agg(F.count("*").alias("views"))
        .orderBy(F.desc("views"))
)

item_freq.show(10)


n_sessions = raw.select("session_id").distinct().count() 
n_items = raw.select("item_id").distinct().count() 
n_pairs = raw.select("session_id", "item_id").distinct().count() 
sparsity = 1 - n_pairs / (n_sessions * n_items)

print(f"Utility matrix: {n_sessions} x {n_items}")
print(f"Observed pairs: {n_pairs}")
print(f"Sparsity: {sparsity:.4%}")


# temporal distribution 
raw.groupBy("month").count().orderBy("month").show() 

raw.groupBy("category").count().orderBy("category").show()

(
    raw.groupBy("session_id", "country")
    .count() 
    .groupBy("country")
    .agg(F.countDistinct("session_id").alias("n_sessions"))
    .orderBy(F.desc("n_sessions"))
    .show(10)
)




+-------+------------------+
|summary|          n_clicks|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
| stddev|   8.9951606320672|
|    min|                 1|
|    max|               195|
+-------+------------------+

Single click session: 5042/24026 (20.985598934487637 %)
+-------+-----+
|item_id|views|
+-------+-----+
|     B4| 3579|
|     A2| 3013|
|    A11| 2789|
|     P1| 2681|
|    B10| 2566|
|     A4| 2522|
|    A15| 2489|
|     A5| 2354|
|    A10| 2280|
|     A1| 2265|
+-------+-----+
only showing top 10 rows

Utility matrix: 24026 x 217
Observed pairs: 148345
Sparsity: 97.1547%
+-----+-----+
|month|count|
+-----+-----+
|    4|48199|
|    5|35654|
|    6|32242|
|    7|35231|
|    8|14148|
+-----+-----+

+--------+-----+
|category|count|
+--------+-----+
|       1|49742|
|       2|38408|
|       3|38577|
|       4|38747|
+--------+-----+

+-------+----------+
|country|n_sessions|
+-------+----------+
|     29|     19582|
|      9|     

In [14]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

w_desc = Window.partitionBy("session_id").orderBy(F.desc("order"))

raw_with_rank = raw.withColumn("click_rank", F.row_number().over(w_desc))

ground_truth_full = (
    raw_with_rank
    .filter(F.col("click_rank") == 1)
    .select("session_id", "item_id")
)

train_raw_full = raw_with_rank.filter(F.col("click_rank") > 1)

session_train_stats = (
    train_raw_full.groupBy("session_id")
        .agg(F.count("*").alias("n_train_clicks"))
)

valid_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") >= 2)
    .select("session_id")
)

cold_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") == 1)
    .select("session_id")
)

train_raw = train_raw_full.join(valid_sessions, on="session_id", how="inner")
ground_truth = ground_truth_full.join(valid_sessions, on="session_id", how="inner")

cold_train_raw = (
    train_raw_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id", "order")
)

cold_ground_truth = (
    ground_truth_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id")
)

print(f"Warm sessions in training: {train_raw.select('session_id').distinct().count()}")
print(f"Warm ground truth pairs:   {ground_truth.count()}")
print(f"Cold-start sessions (1 observed click): {cold_sessions.count()}")
print(f"Cold ground-truth pairs:               {cold_ground_truth.count()}")



Warm sessions in training: 15664
Warm ground truth pairs:   15664
Cold-start sessions (1 observed click): 3320
Cold ground-truth pairs:               3320


In [15]:
# import itertools

# ranks = [10, 20, 50]
# regParams = [0.1, 1.0]
# alphas = [40.0, 80.0]

# best_ndcg = -1
# best_params = None

# all_test_sessions = ground_truth_idx.select("session_idx").distinct()

# print("Grid search on NDCG@10:")
# for r, reg, a in itertools.product(ranks, regParams, alphas):
#     als_tune = ALS(
#         rank=r, maxIter=10, regParam=reg, alpha=a,
#         implicitPrefs=True, userCol="session_idx", itemCol="item_idx", ratingCol="click_count",
#         coldStartStrategy="drop", seed=42
#     )
#     model_tune = als_tune.fit(train_idx)
#     raw_recs_tune = model_tune.recommendForUserSubset(all_test_sessions, numItems=10)
    
#     recs_tune = (
#         raw_recs_tune
#         .select("session_idx", F.posexplode("recommendations").alias("rank_0", "rec"))
#         .select("session_idx", (F.col("rank_0") + 1).alias("rank"), F.col("rec.item_idx").alias("item_idx"))
#     )
    
#     metrics_tune = evaluate_recommendations(recs_tune, ground_truth_idx, k=10)
#     cur_ndcg = metrics_tune["NDCG@10"]
#     print(f"  rank={r}, reg={reg}, alpha={a} -> NDCG@10={cur_ndcg:.4f}")
    
#     if cur_ndcg > best_ndcg: 
#         best_ndcg = cur_ndcg
#         best_params = (r, reg, a)

# print(f"\nBest params: rank={best_params[0]}, reg={best_params[1]}, alpha={best_params[2]} (NDCG@10={best_ndcg:.4f})")

In [16]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS

TOP_K = 10
ALS_RANK = 20
ALS_ALPHA = 80.0
ALS_REG_PARAM = 1.0
ALS_NUM_ITEMS = 217

interactions = (
    train_raw
        .groupBy("session_id", "item_id")
        .agg(F.count("*").alias("click_count"))
)

item_indexer = StringIndexer(inputCol='item_id', outputCol="item_idx").fit(interactions)

train_idx = (
    item_indexer.transform(interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

train_idx.cache()

ground_truth_idx = (
    item_indexer.transform(ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .select("session_idx", "item_idx")
)

als = ALS(
    rank=ALS_RANK,
    maxIter=15,
    regParam=ALS_REG_PARAM,
    alpha=ALS_ALPHA,
    implicitPrefs=True,
    userCol="session_idx",
    itemCol="item_idx",
    ratingCol="click_count",
    coldStartStrategy="drop",
    seed=42,
)

als_model = als.fit(train_idx)

all_sessions = ground_truth_idx.select("session_idx").distinct()
raw_recs = als_model.recommendForUserSubset(all_sessions, numItems=ALS_NUM_ITEMS)

als_recs_full = (
    raw_recs
    .select(
        F.col("session_idx"),
        F.posexplode("recommendations").alias("rank_0", "rec")
    )
    .select(
        "session_idx",
        (F.col("rank_0") + 1).alias("rank"),
        F.col("rec.item_idx").alias("item_idx"),
        F.col("rec.rating").alias("score"),
    )
)

print(f"ALS recommendation rows before filtering: {als_recs_full.count()}")
als_recs_full.show(10)


ALS recommendation rows before filtering: 3399088


+-----------+----+--------+----------+
|session_idx|rank|item_idx|     score|
+-----------+----+--------+----------+
|          1|   1|      31| 1.0824772|
|          1|   2|      29| 1.0234141|
|          1|   3|      79|0.97137964|
|          1|   4|       6| 0.9692418|
|          1|   5|     135|0.94846284|
|          1|   6|      17|0.94593084|
|          1|   7|       0|0.94507164|
|          1|   8|     143|0.90349835|
|          1|   9|     129|0.88752854|
|          1|  10|      28|0.87011105|
+-----------+----+--------+----------+
only showing top 10 rows



In [17]:
def exclude_seen_items(recs_df, seen_items_df, score_col="score", k=TOP_K):
    filtered = (
        recs_df.alias("c")
        .join(
            seen_items_df.alias("seen"),
            (F.col("c.session_idx") == F.col("seen.session_idx"))
            & (F.col("c.item_idx") == F.col("seen.seen_item_idx")),
            how="left_anti",
        )
    )

    return (
        filtered
        .withColumn(
            "rank",
            F.row_number().over(
                Window.partitionBy("session_idx").orderBy(F.desc(score_col), F.asc("item_idx"))
            ),
        )
        .filter(F.col("rank") <= k)
    )


def evaluate_recommendations(recs_df, ground_truth_df, k=TOP_K):
    eval_sessions = ground_truth_df.select("session_idx").distinct()

    top_k = (
        recs_df.filter(F.col("rank") <= k)
        .withColumn(
            "dedupe_rank",
            F.row_number().over(
                Window.partitionBy("session_idx", "item_idx").orderBy(F.col("rank"))
            ),
        )
        .filter(F.col("dedupe_rank") == 1)
        .drop("dedupe_rank")
    )

    hits = (
        top_k.join(
            ground_truth_df.withColumn("hit", F.lit(1.0)),
            on=["session_idx", "item_idx"],
            how="left",
        )
        .fillna(0, subset=["hit"])
    )

    n_rel = (
        ground_truth_df.groupBy("session_idx")
        .agg(F.count("*").alias("n_relevant"))
    )

    dcg = (
        hits
        .withColumn(
            "gain",
            (F.pow(F.lit(2.0), F.col("hit")) - 1.0)
            / F.log2(F.col("rank").cast("double") + 1.0),
        )
        .groupBy("session_idx")
        .agg(F.sum("gain").alias("dcg"))
    )

    idcg = (
        n_rel.withColumn("k_eff", F.least(F.col("n_relevant"), F.lit(k)))
        .withColumn(
            "idcg",
            F.expr(
                "aggregate(sequence(1, k_eff), 0D, (a, i) -> a + 1.0 / log2(cast(i + 1 as double)))"
            ),
        )
    )

    ndcg_scores = (
        eval_sessions
        .join(dcg, on="session_idx", how="left")
        .fillna(0.0, subset=["dcg"])
        .join(idcg, on="session_idx", how="left")
        .withColumn(
            "ndcg",
            F.when(F.col("idcg") > 0, F.col("dcg") / F.col("idcg")).otherwise(0.0),
        )
    )

    first_hits = (
        hits.filter(F.col("hit") == 1.0)
        .groupBy("session_idx")
        .agg(F.min("rank").alias("first_hit_rank"))
    )

    mrr_scores = (
        eval_sessions
        .join(first_hits, on="session_idx", how="left")
        .withColumn(
            "rr",
            F.when(F.col("first_hit_rank").isNotNull(), 1.0 / F.col("first_hit_rank")).otherwise(0.0),
        )
    )

    pr = (
        eval_sessions
        .join(hits.groupBy("session_idx").agg(F.sum("hit").alias("n_hits")), on="session_idx", how="left")
        .fillna(0.0, subset=["n_hits"])
        .join(n_rel, on="session_idx", how="left")
        .fillna(0.0, subset=["n_relevant"])
        .withColumn("precision", F.col("n_hits") / F.lit(k))
        .withColumn(
            "recall",
            F.when(F.col("n_relevant") > 0, F.col("n_hits") / F.col("n_relevant")).otherwise(0.0),
        )
    )

    results = {
        f"NDCG@{k}": ndcg_scores.agg(F.mean("ndcg")).collect()[0][0],
        f"MRR@{k}": mrr_scores.agg(F.mean("rr")).collect()[0][0],
        f"Precision@{k}": pr.agg(F.mean("precision")).collect()[0][0],
        f"Recall@{k}": pr.agg(F.mean("recall")).collect()[0][0],
    }
    return results


In [18]:
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, Normalizer
from pyspark.ml.functions import vector_to_array

item_attributes = (
    raw.groupBy("item_id")
    .agg(
        F.first("category").alias("category"),
        F.first("colour").alias("colour"),
        F.first("photo_type").alias("photo_type"),
        F.first("price_above_avg").alias("price_above_avg"),
        F.avg("price").alias("avg_price"),
    )
)

price_stats = item_attributes.agg(
    F.min("avg_price").alias("min_price"),
    F.max("avg_price").alias("max_price"),
).collect()[0]
price_min = float(price_stats["min_price"])
price_max = float(price_stats["max_price"])
price_range = price_max - price_min if price_max > price_min else 1.0

item_features = (
    item_indexer.transform(item_attributes)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("avg_price_scaled", (F.col("avg_price") - F.lit(price_min)) / F.lit(price_range))
)

ohe = OneHotEncoder(
    inputCols=["category", "colour", "photo_type", "price_above_avg"],
    outputCols=["cat_vec", "col_vec", "photo_vec", "price_vec"],
    dropLast=True,
)
ohe_model = ohe.fit(item_features)

assembler = VectorAssembler(
    inputCols=["cat_vec", "col_vec", "photo_vec", "price_vec", "avg_price_scaled"],
    outputCol="content_features_raw"
)
normalizer = Normalizer(inputCol="content_features_raw", outputCol="content_features", p=2.0)

item_content = (
    normalizer
    .transform(assembler.transform(ohe_model.transform(item_features)))
    .select("item_idx", vector_to_array("content_features").alias("feature_arr"))
)


In [19]:
item_popularity = (
    train_idx
    .groupBy("item_idx")
    .agg(F.sum("click_count").alias("score"))
)

warm_seen_items = train_idx.select("session_idx", F.col("item_idx").alias("seen_item_idx")).distinct()

pop_recs = (
    ground_truth_idx.select("session_idx").distinct()
    .crossJoin(item_popularity)
    .transform(lambda df: exclude_seen_items(df, warm_seen_items, score_col="score", k=TOP_K))
)

als_recs = exclude_seen_items(als_recs_full, warm_seen_items, score_col="score", k=TOP_K)

pop_metrics = evaluate_recommendations(pop_recs, ground_truth_idx, k=TOP_K)
als_metrics = evaluate_recommendations(als_recs, ground_truth_idx, k=TOP_K)

def diversity_metrics(recs_df, train_df, n_catalog_items=217):
    n_unique_recommended = recs_df.select("item_idx").distinct().count()
    coverage = n_unique_recommended / n_catalog_items

    return {
        "Coverage": coverage,
        "UniqueRecommendedItems": n_unique_recommended,
    }

als_div = diversity_metrics(als_recs, train_idx, n_catalog_items=217)
pop_div = diversity_metrics(pop_recs, train_idx, n_catalog_items=217)

print(f"Popularity Coverage: {pop_div['UniqueRecommendedItems']}/217 = {pop_div['Coverage']:.2%}")
print(f"ALS Coverage:        {als_div['UniqueRecommendedItems']}/217 = {als_div['Coverage']:.2%}")

print(f"NDCG@10      baseline={pop_metrics['NDCG@10']:.4f}  ALS={als_metrics['NDCG@10']:.4f}  delta={als_metrics['NDCG@10'] - pop_metrics['NDCG@10']:+.4f}")
print(f"Precision@10 baseline={pop_metrics['Precision@10']:.4f}  ALS={als_metrics['Precision@10']:.4f}  delta={als_metrics['Precision@10'] - pop_metrics['Precision@10']:+.4f}")
print(f"Recall@10    baseline={pop_metrics['Recall@10']:.4f}  ALS={als_metrics['Recall@10']:.4f}  delta={als_metrics['Recall@10'] - pop_metrics['Recall@10']:+.4f}")
print(f"MRR@10       baseline={pop_metrics['MRR@10']:.4f}  ALS={als_metrics['MRR@10']:.4f}  delta={als_metrics['MRR@10'] - pop_metrics['MRR@10']:+.4f}")
print(f"Coverage     baseline={pop_div['Coverage']:.2%}  ALS={als_div['Coverage']:.2%}  delta={als_div['Coverage'] - pop_div['Coverage']:+.2%}")


Popularity Coverage: 50/217 = 23.04%
ALS Coverage:        212/217 = 97.70%
NDCG@10      baseline=0.0512  ALS=0.1110  delta=+0.0597
Precision@10 baseline=0.0103  ALS=0.0222  delta=+0.0119
Recall@10    baseline=0.1027  ALS=0.2221  delta=+0.1194
MRR@10       baseline=0.0360  ALS=0.0775  delta=+0.0416
Coverage     baseline=23.04%  ALS=97.70%  delta=+74.65%


In [20]:
# content-based filtering: score candidates by similarity to the items observed in the current session
from pyspark.sql.functions import col

KNN_NEIGHBORS = 50

item_item_sim = (
    item_content.alias("a")
    .crossJoin(item_content.alias("b"))
    .filter(col("a.item_idx") != col("b.item_idx"))
    .select(
        F.col("a.item_idx").alias("seed_idx"),
        F.col("b.item_idx").alias("target_idx"),
        F.expr(
            "aggregate(zip_with(a.feature_arr, b.feature_arr, (x, y) -> x * y), 0D, (acc, x) -> acc + x)"
        ).alias("similarity"),
    )
)

item_item_sim = (
    item_item_sim
    .withColumn(
        "neighbor_rank",
        F.row_number().over(
            Window.partitionBy("seed_idx").orderBy(F.desc("similarity"), F.asc("target_idx"))
        ),
    )
    .filter((F.col("neighbor_rank") <= KNN_NEIGHBORS) & (F.col("similarity") > 0))
    .drop("neighbor_rank")
)

test_history = train_idx.join(ground_truth_idx.select("session_idx").distinct(), on="session_idx", how="inner")
seen_items = test_history.select("session_idx", F.col("item_idx").alias("seen_item_idx")).distinct()

knn_candidate_scores = (
    test_history.alias("h")
    .join(item_item_sim.alias("s"), col("h.item_idx") == col("s.seed_idx"))
    .groupBy(F.col("h.session_idx").alias("session_idx"), F.col("s.target_idx").alias("item_idx"))
    .agg(F.sum(col("h.click_count") * col("s.similarity")).alias("score"))
)

knn_recs = exclude_seen_items(knn_candidate_scores, seen_items, score_col="score", k=TOP_K)

knn_metrics = evaluate_recommendations(knn_recs, ground_truth_idx, k=TOP_K)
print("Content-based k-NN")
for k_m, v_m in knn_metrics.items():
    print(f"  {k_m}: {v_m:.4f}")

knn_div = diversity_metrics(knn_recs, train_idx, n_catalog_items=217)
print(f"Content k-NN Coverage: {knn_div['UniqueRecommendedItems']}/217 = {knn_div['Coverage']:.2%}")


Content-based k-NN
  NDCG@10: 0.0538
  MRR@10: 0.0363
  Precision@10: 0.0113
  Recall@10: 0.1128


Content k-NN Coverage: 215/217 = 99.08%


In [21]:
# cold sessions have only 1 observed item in training

cold_interactions = (
    cold_train_raw.groupBy("session_id", "item_id")
    .agg(F.count("*").alias("click_count"))
)

cold_train_idx = (
    item_indexer.transform(cold_interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

cold_ground_truth_idx = (
    item_indexer.transform(cold_ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

cold_seen_items = cold_train_idx.select("session_idx", F.col("item_idx").alias("seen_item_idx")).distinct()

pop_recs_cold = (
    cold_ground_truth_idx.select("session_idx").distinct()
    .crossJoin(item_popularity)
    .transform(lambda df: exclude_seen_items(df, cold_seen_items, score_col="score", k=TOP_K))
)

pop_metrics_cold = evaluate_recommendations(pop_recs_cold, cold_ground_truth_idx, k=TOP_K)
pop_div_cold = diversity_metrics(pop_recs_cold, train_idx, n_catalog_items=217)

seq_candidate_scores = (
    cold_train_idx.alias("h")
    .join(item_item_sim.alias("s"), col("h.item_idx") == col("s.seed_idx"))
    .groupBy(F.col("h.session_idx").alias("session_idx"), F.col("s.target_idx").alias("item_idx"))
    .agg(F.sum(col("h.click_count") * col("s.similarity")).alias("score"))
)

seq_recs = exclude_seen_items(seq_candidate_scores, cold_seen_items, score_col="score", k=TOP_K)

seq_metrics = evaluate_recommendations(seq_recs, cold_ground_truth_idx, k=TOP_K)
seq_div = diversity_metrics(seq_recs, train_idx, n_catalog_items=217)

print(f"Cold-start Evaluation (N={cold_ground_truth_idx.select('session_idx').distinct().count()} sessions):")
print(f"  Popularity NDCG@10: {pop_metrics_cold['NDCG@10']:.4f}")
print(f"  Content Seq-Warmup NDCG@10: {seq_metrics['NDCG@10']:.4f}")


Cold-start Evaluation (N=3320 sessions):
  Popularity NDCG@10: 0.0807
  Content Seq-Warmup NDCG@10: 0.0858


In [ ]:
# Final evaluation results
evaluation_metrics = {
    "warm_sessions": {
        "popularity": {
            "NDCG@10": round(pop_metrics["NDCG@10"], 4),
            "MRR@10": round(pop_metrics["MRR@10"], 4),
            "Recall@10": round(pop_metrics["Recall@10"], 4),
            "Coverage": round(pop_div["Coverage"], 4),
        },
        "als": {
            "NDCG@10": round(als_metrics["NDCG@10"], 4),
            "MRR@10": round(als_metrics["MRR@10"], 4),
            "Recall@10": round(als_metrics["Recall@10"], 4),
            "Coverage": round(als_div["Coverage"], 4),
        },
        "knn": {
            "NDCG@10": round(knn_metrics["NDCG@10"], 4),
            "MRR@10": round(knn_metrics["MRR@10"], 4),
            "Recall@10": round(knn_metrics["Recall@10"], 4),
            "Coverage": round(knn_div["Coverage"], 4),
        }
    },
    "cold_sessions": {
        "popularity": {
            "NDCG@10": round(pop_metrics_cold["NDCG@10"], 4),
            "Recall@10": round(pop_metrics_cold["Recall@10"], 4),
            "Coverage": round(pop_div_cold["Coverage"], 4),
        },
        "sequential_warmup": {
            "NDCG@10": round(seq_metrics["NDCG@10"], 4),
            "Recall@10": round(seq_metrics["Recall@10"], 4),
            "Coverage": round(seq_div["Coverage"], 4),
        }
    }
}

print("\n---Final evaluation metrics---")
print(evaluation_metrics)

import json
import os

top_items_list = (
    item_popularity.orderBy(F.desc("score"), F.asc("item_idx"))
    .limit(10)
    .select("item_idx", "score")
    .collect()
)
labels = item_indexer.labels
top_items_data = [
    {"item_id": labels[int(row.item_idx)], "views": int(row.score)}
    for row in top_items_list
]

als_params = {
    "rank": ALS_RANK,
    "alpha": ALS_ALPHA,
    "regParam": ALS_REG_PARAM
}

output_path = "../evaluation_metrics/recommender_metrics.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

full_metrics = {
    "metrics": evaluation_metrics,
    "top_items": top_items_data,
    "params": als_params
}

with open(output_path, "w") as f:
    json.dump(full_metrics, f, indent=4)

print(f"Metrics saved to {output_path}")



---Final evaluation metrics---
{'warm_sessions': {'popularity': {'NDCG@10': 0.0512, 'MRR@10': 0.036, 'Recall@10': 0.1027, 'Coverage': 0.2304}, 'als': {'NDCG@10': 0.111, 'MRR@10': 0.0775, 'Recall@10': 0.2221, 'Coverage': 0.977}, 'knn': {'NDCG@10': 0.0538, 'MRR@10': 0.0363, 'Recall@10': 0.1128, 'Coverage': 0.9908}}, 'cold_sessions': {'popularity': {'NDCG@10': 0.0807, 'Recall@10': 0.1623, 'Coverage': 0.0507}, 'sequential_warmup': {'NDCG@10': 0.0858, 'Recall@10': 0.1717, 'Coverage': 0.9908}}}
Metrics saved to ../output/recommender_metrics.json
